In [ ]:
# ===================================================================
# ==     SCRIPT 1: TRI-CLASSIFICATION (AUGMENTED) MobileNetV2      ==
# ===================================================================

!pip install -q matplotlib seaborn scikit-learn tensorflow

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from itertools import cycle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score
)

plt.style.use('default')
COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

# --- 1. Environment and Configuration ---
DRIVE_MODEL_SAVE_PATH = '/content/drive/MyDrive/summer_2025_research'
try:
    base_dir = os.path.join(DRIVE_MODEL_SAVE_PATH, 'Dataset/dataset')
    os.makedirs(DRIVE_MODEL_SAVE_PATH, exist_ok=True)
    os.listdir(base_dir)
except FileNotFoundError:
    print("Google Drive not found or dataset path incorrect. Using a local placeholder path.")
    base_dir = 'dataset_placeholder'
    DRIVE_MODEL_SAVE_PATH = '.'
    for c in ['cn', 'emci', 'lmci']:
        os.makedirs(os.path.join(base_dir, c), exist_ok=True)

# Model & Training Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
MODEL_NAME = 'MobileNetV2'
OPTIMIZER_NAME = 'Adam'
CLASSES = ['cn', 'emci', 'lmci']
NUM_CLASSES = len(CLASSES)
LOSS_FUNCTION = 'categorical_crossentropy'
TARGET_INFLATION_PER_CLASS = 2000

# --- 2. Data Loading ---
filepaths, labels = [], []
for cls in CLASSES:
    class_dir = os.path.join(base_dir, cls)
    if not os.path.exists(class_dir): continue
    for filename in os.listdir(class_dir):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(class_dir, filename))
            labels.append(cls)
df = pd.DataFrame({'filepath': filepaths, 'label': labels})

# --- 3. Functions for Training and Evaluation ---
def build_model():
    """Builds and compiles a new instance of the tri-classification model."""
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions_layer = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions_layer)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=LOSS_FUNCTION, metrics=['accuracy'])
    return model

def plot_training_history(history, cycle_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].plot(history.history['accuracy'], color=COLORS['purple'], linewidth=2, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Accuracy')
    axes[0].set_title(f'Model Accuracy ({cycle_name})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['loss'], color=COLORS['purple'], linewidth=2, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Loss')
    axes[1].set_title(f'Model Loss ({cycle_name})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def plot_multi_class_roc(y_true, y_pred_proba, cycle_name):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"], tpr["macro"], roc_auc["macro"] = all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})', color='navy', linestyle=':', linewidth=4)
    colors = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'ROC of class {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12); plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'Multi-Class ROC Curve ({cycle_name})', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.show()

def run_training_and_evaluation(train_df, val_df, test_df, cycle_name):
    print(f"\n{'='*40}\n  STARTING CYCLE: {cycle_name}\n{'='*40}")

    train_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_preprocess,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        fill_mode='nearest'
    )
    val_test_datagen = ImageDataGenerator(preprocessing_function=mobilenet_preprocess)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=True
    )
    validation_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )
    test_generator = val_test_datagen.flow_from_dataframe(
        dataframe=test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )

    model = build_model()
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    history = model.fit(train_generator, epochs=EPOCHS, validation_data=validation_generator, callbacks=[early_stopping], verbose=1)

    print(f"\n--- Performance on Test Set ({cycle_name}) ---")
    predictions_prob = model.predict(test_generator)
    y_pred = np.argmax(predictions_prob, axis=1)
    y_true = test_generator.classes

    print("\nClassification Report:"); print(classification_report(y_true, y_pred, target_names=CLASSES))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'Confusion Matrix ({cycle_name})', fontweight='bold'); plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.show()

    plot_training_history(history, cycle_name)
    plot_multi_class_roc(y_true, predictions_prob, cycle_name)
    return model

# --- 4. Main Execution Logic (FIXED & SIMPLIFIED) ---
if __name__ == "__main__":
    if df.empty:
        raise ValueError("Dataset is empty. Check data paths.")

    # --- Dataset Breakdown ---
    print("\n--- Dataset Breakdown ---")
    print("1. Original Image Counts (Before any splitting):")
    print(df['label'].value_counts().to_string())
    print("-" * 35)

    # --- Step 1: Split the original data to create pristine validation and test sets ---
    train_val_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
    train_df, val_df = train_test_split(train_val_df, test_size=0.20, random_state=42, stratify=train_val_df['label'])

    print("2. Data Splitting (From Original Dataset):")
    print(f"  - Initial Training Set (before inflation): {len(train_df)} images")
    print(f"  - Validation Set:                          {len(val_df)} images")
    print(f"  - Test Set:                                {len(test_df)} images")
    print("-" * 35)

    # --- Step 2: Inflate ONLY the training set by oversampling each class ---
    print(f"3. Inflating Training Set (Oversampling to {TARGET_INFLATION_PER_CLASS} samples per class):")
    inflated_dfs = []
    for cls in CLASSES:
        class_subset = train_df[train_df['label'] == cls]
        inflated_dfs.append(class_subset.sample(n=TARGET_INFLATION_PER_CLASS, replace=True, random_state=42))

    inflated_train_df = pd.concat(inflated_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    print("-" * 35)

    # --- Final Summary ---
    print("4. Final Dataset Sizes for Model:")
    print(f"  - Inflated Training Set:   {len(inflated_train_df)} images")
    print(inflated_train_df['label'].value_counts().to_string())
    print(f"\n  - Validation Set (Pristine): {len(val_df)} images")
    print(val_df['label'].value_counts().to_string())
    print(f"\n  - Test Set (Pristine):         {len(test_df)} images")
    print(test_df['label'].value_counts().to_string())
    print("-" * 35)

    # --- Run Training Cycle ---
    augmented_model = run_training_and_evaluation(
        train_df=inflated_train_df,
        val_df=val_df,
        test_df=test_df,
        cycle_name="Training with Augmented & Inflated Data"
    )

    # --- Model Saving ---
    save_dir = "/content/drive/MyDrive/summer_2025_research/bestmodel/MobileNetV2_tri"
    os.makedirs(save_dir, exist_ok=True)
    model_save_path = os.path.join(save_dir, "TRAINING_WITH_INFLATED_DATA_TRI_75.keras")
    augmented_model.save(model_save_path)
    print(f"\n Model saved successfully to: {model_save_path}")

In [ ]:
# ===================================================================
# ==      SCRIPT 2: TRI-CLASSIFICATION (ORIGINAL) MobileNetV2      ==
# ===================================================================

!pip install -q matplotlib seaborn scikit-learn tensorflow

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from itertools import cycle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score
)

plt.style.use('default')
COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

# --- 1. Environment and Configuration ---
DRIVE_MODEL_SAVE_PATH = '/content/drive/MyDrive/summer_2025_research'
try:
    base_dir = os.path.join(DRIVE_MODEL_SAVE_PATH, 'Dataset/dataset')
    os.makedirs(DRIVE_MODEL_SAVE_PATH, exist_ok=True)
    os.listdir(base_dir)
except FileNotFoundError:
    print("Google Drive not found or dataset path incorrect. Using a local placeholder path.")
    base_dir = 'dataset_placeholder'
    DRIVE_MODEL_SAVE_PATH = '.'
    for c in ['cn', 'emci', 'lmci']:
        os.makedirs(os.path.join(base_dir, c), exist_ok=True)

# Model & Training Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
MODEL_NAME = 'MobileNetV2'
OPTIMIZER_NAME = 'Adam'
CLASSES = ['cn', 'emci', 'lmci']
NUM_CLASSES = len(CLASSES)
LOSS_FUNCTION = 'categorical_crossentropy'

# --- 2. Data Loading ---
filepaths, labels = [], []
for cls in CLASSES:
    class_dir = os.path.join(base_dir, cls)
    if not os.path.exists(class_dir): continue
    for filename in os.listdir(class_dir):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(class_dir, filename))
            labels.append(cls)
df = pd.DataFrame({'filepath': filepaths, 'label': labels})

# --- 3. Functions for Training and Evaluation ---
def build_model():
    """Builds and compiles a new instance of the tri-classification model."""
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions_layer = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions_layer)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=LOSS_FUNCTION, metrics=['accuracy'])
    return model

def plot_training_history(history, cycle_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].plot(history.history['accuracy'], color=COLORS['purple'], linewidth=2, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Accuracy')
    axes[0].set_title(f'Model Accuracy ({cycle_name})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['loss'], color=COLORS['purple'], linewidth=2, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Loss')
    axes[1].set_title(f'Model Loss ({cycle_name})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def plot_multi_class_roc(y_true, y_pred_proba, cycle_name):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"], tpr["macro"], roc_auc["macro"] = all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})', color='navy', linestyle=':', linewidth=4)
    colors = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'ROC of class {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12); plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'Multi-Class ROC Curve ({cycle_name})', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.show()

def run_training_and_evaluation(train_df, val_df, test_df, cycle_name):
    print(f"\n{'='*40}\n  STARTING CYCLE: {cycle_name}\n{'='*40}")

    datagen = ImageDataGenerator(rescale=1./255)

    train_generator = datagen.flow_from_dataframe(
        dataframe=train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=True
    )
    validation_generator = datagen.flow_from_dataframe(
        dataframe=val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )
    test_generator = datagen.flow_from_dataframe(
        dataframe=test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )

    model = build_model()
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    history = model.fit(train_generator, epochs=EPOCHS, validation_data=validation_generator, callbacks=[early_stopping], verbose=1)

    print(f"\n--- Performance on Test Set ({cycle_name}) ---")
    predictions_prob = model.predict(test_generator)
    y_pred = np.argmax(predictions_prob, axis=1)
    y_true = test_generator.classes

    print("\nClassification Report:"); print(classification_report(y_true, y_pred, target_names=CLASSES))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'Confusion Matrix ({cycle_name})', fontweight='bold'); plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.show()

    plot_training_history(history, cycle_name)
    plot_multi_class_roc(y_true, predictions_prob, cycle_name)
    return model

# --- 4. Main Execution Logic ---
if __name__ == "__main__":
    if df.empty:
        raise ValueError("Dataset is empty. Check data paths.")

    # --- Dataset Breakdown ---
    print("\n--- Dataset Breakdown ---")
    print("1. Original Image Counts (Imbalanced):")
    original_counts = df['label'].value_counts()
    print(original_counts.to_string())
    print("-" * 35)

    # --- Step 1: Balancing by Downsampling ---
    min_samples = original_counts.min()
    balanced_df = pd.concat([
        df[df['label'] == cls].sample(n=min_samples, random_state=42) for cls in CLASSES
    ]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"2. After Balancing (Downsampling all classes to {min_samples} samples):")
    print(f"  - Total balanced samples: {len(balanced_df)} images")
    print(balanced_df['label'].value_counts().to_string())
    print("-" * 35)

    # --- Step 2: Splitting the Balanced Data ---
    train_val_df, test_df = train_test_split(balanced_df, test_size=0.20, random_state=42, stratify=balanced_df['label'])
    train_df, val_df = train_test_split(train_val_df, test_size=0.20, random_state=42, stratify=train_val_df['label'])

    print("3. Final Dataset Sizes for Model:")
    print(f"  - Training Set:   {len(train_df)} images")
    print(train_df['label'].value_counts().to_string())
    print(f"\n  - Validation Set: {len(val_df)} images")
    print(val_df['label'].value_counts().to_string())
    print(f"\n  - Test Set:         {len(test_df)} images")
    print(test_df['label'].value_counts().to_string())
    print("-" * 35)

    print("\n4. ❕ Note on Data Augmentation:")
    print("This script does NOT use data augmentation. All images in the")
    print("training, validation, and test sets are only rescaled.")

    # --- Run Training Cycle ---
    original_model = run_training_and_evaluation(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        cycle_name="Training on Original Balanced Data"
    )

    # --- Model Saving ---
    save_dir = "/content/drive/MyDrive/summer_2025_research/bestmodel/MobileNetV2_tri"
    os.makedirs(save_dir, exist_ok=True)
    model_save_path = os.path.join(save_dir, "TRAINING_WITH_ORIGINAL_DATA_TRI.keras")
    original_model.save(model_save_path)
    print(f"\n Model saved successfully to: {model_save_path}")

In [ ]:
# ==============================================================================
# ==      SCRIPT 1: TRI-CLASSIFICATION (AUGMENTED) EfficientNetV2B0           ==
# ==============================================================================

!pip install -q matplotlib seaborn scikit-learn tensorflow

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from itertools import cycle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as efficientnet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score
)

plt.style.use('default')
COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

# --- 1. Environment and Configuration ---
DRIVE_MODEL_SAVE_PATH = '/content/drive/MyDrive/summer_2025_research'
try:
    base_dir = os.path.join(DRIVE_MODEL_SAVE_PATH, 'Dataset/dataset')
    os.makedirs(DRIVE_MODEL_SAVE_PATH, exist_ok=True)
    os.listdir(base_dir)
except FileNotFoundError:
    print("Google Drive not found or dataset path incorrect. Using a local placeholder path.")
    base_dir = 'dataset_placeholder'
    DRIVE_MODEL_SAVE_PATH = '.'
    for c in ['cn', 'emci', 'lmci']:
        os.makedirs(os.path.join(base_dir, c), exist_ok=True)

# Model & Training Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
MODEL_NAME = 'EfficientNetV2B0'
OPTIMIZER_NAME = 'Adam'
LOSS_FUNCTION = 'categorical_crossentropy' # Changed for multi-class
CLASSES = ['cn', 'emci', 'lmci'] # Changed for multi-class
NUM_CLASSES = len(CLASSES)
TARGET_SAMPLES_PER_CLASS = 2000 # Target for oversampling

# --- 2. Data Loading ---
filepaths, labels = [], []
for cls in CLASSES:
    class_dir = os.path.join(base_dir, cls)
    if not os.path.exists(class_dir): continue
    for filename in os.listdir(class_dir):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(class_dir, filename))
            labels.append(cls) # Use the direct class name

df = pd.DataFrame({'filepath': filepaths, 'label': labels})

# --- 3. Functions for Training and Evaluation ---
def build_model():
    """Builds and compiles a new instance of the tri-class classification model."""
    base_model = EfficientNetV2B0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions_layer = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions_layer)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=LOSS_FUNCTION, metrics=['accuracy'])
    return model

def plot_training_history(history, cycle_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].plot(history.history['accuracy'], color=COLORS['purple'], linewidth=2, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Accuracy')
    axes[0].set_title(f'Model Accuracy ({cycle_name})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['loss'], color=COLORS['purple'], linewidth=2, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Loss')
    axes[1].set_title(f'Model Loss ({cycle_name})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def plot_multi_class_roc(y_true, y_pred_proba, cycle_name):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"], tpr["macro"], roc_auc["macro"] = all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})', color='navy', linestyle=':', linewidth=4)
    colors = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'ROC of class {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12); plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'Multi-Class ROC Curve ({cycle_name})', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.show()

def run_training_and_evaluation(train_df, val_df, test_df, cycle_name):
    print(f"\n{'='*40}\n  STARTING CYCLE: {cycle_name}\n{'='*40}")

    # Data augmentation for the training set
    train_datagen = ImageDataGenerator(
        preprocessing_function=efficientnet_preprocess,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        fill_mode='nearest'
    )
    # Only preprocessing for validation and test sets
    val_test_datagen = ImageDataGenerator(preprocessing_function=efficientnet_preprocess)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=True
    )
    validation_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )
    test_generator = val_test_datagen.flow_from_dataframe(
        dataframe=test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )

    model = build_model()
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    history = model.fit(train_generator, epochs=EPOCHS, validation_data=validation_generator, callbacks=[early_stopping], verbose=1)

    print(f"\n--- Performance on Test Set ({cycle_name}) ---")
    predictions_prob = model.predict(test_generator)
    y_pred = np.argmax(predictions_prob, axis=1) # Get class with highest probability
    y_true = test_generator.classes

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'Confusion Matrix ({cycle_name})', fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.show()

    plot_training_history(history, cycle_name)
    plot_multi_class_roc(y_true, predictions_prob, cycle_name)
    return model

# --- 4. Main Execution Logic ---
if __name__ == "__main__":
    if df.empty:
        raise ValueError("Dataset is empty. Check data paths.")

    print("\n--- Dataset Breakdown ---")
    print("1. Original Image Counts (Before any splitting):")
    print(df['label'].value_counts().to_string())
    print("-" * 35)

    # Step 1: Split the original data to create pristine validation and test sets
    train_val_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
    train_df, val_df = train_test_split(train_val_df, test_size=0.20, random_state=42, stratify=train_val_df['label'])

    # Step 2: Inflate ONLY the training set by oversampling each class
    print("2. Inflating Training Set via Oversampling:")
    inflated_dfs = []
    for cls in CLASSES:
        class_df = train_df[train_df['label'] == cls]
        inflated_dfs.append(class_df.sample(n=TARGET_SAMPLES_PER_CLASS, replace=True, random_state=42))

    final_train_df = pd.concat(inflated_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

    # Final breakdown of the datasets being sent to the model
    print("\n3. Final Dataset Sizes for Model:")
    print(f"  - Inflated Training Set:   {len(final_train_df)} images")
    print(final_train_df['label'].value_counts().to_string())
    print(f"\n  - Validation Set (Pristine): {len(val_df)} images")
    print(val_df['label'].value_counts().to_string())
    print(f"\n  - Test Set (Pristine):         {len(test_df)} images")
    print(test_df['label'].value_counts().to_string())
    print("-" * 35)

    # --- Run Training Cycle ---
    augmented_model = run_training_and_evaluation(
        train_df=final_train_df,
        val_df=val_df,
        test_df=test_df,
        cycle_name="Training with Inflated & Augmented Data"
    )

    # --- Model Saving ---
    save_dir = "/content/drive/MyDrive/summer_2025_research/bestmodel/EfficientNetV2B0_tri"
    os.makedirs(save_dir, exist_ok=True)
    model_save_path = os.path.join(save_dir, "TRAINING_WITH_INFLATED_AUGMENTED_DATA_EFV2B0.keras")
    augmented_model.save(model_save_path)
    print(f"\n Model saved successfully to: {model_save_path}")

In [ ]:
# ==============================================================================
# ==      SCRIPT 2: TRI-CLASSIFICATION (ORIGINAL)   EfficientNetV2B0          ==
# ==============================================================================

!pip install -q matplotlib seaborn scikit-learn tensorflow

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from itertools import cycle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as efficientnet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score
)

plt.style.use('default')
COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

# --- 1. Environment and Configuration ---
DRIVE_MODEL_SAVE_PATH = '/content/drive/MyDrive/summer_2025_research'
try:
    base_dir = os.path.join(DRIVE_MODEL_SAVE_PATH, 'Dataset/dataset')
    os.makedirs(DRIVE_MODEL_SAVE_PATH, exist_ok=True)
    os.listdir(base_dir)
except FileNotFoundError:
    print("Google Drive not found or dataset path incorrect. Using a local placeholder path.")
    base_dir = 'dataset_placeholder'
    DRIVE_MODEL_SAVE_PATH = '.'
    for c in ['cn', 'emci', 'lmci']:
        os.makedirs(os.path.join(base_dir, c), exist_ok=True)

# Model & Training Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
MODEL_NAME = 'EfficientNetV2B0'
OPTIMIZER_NAME = 'Adam'
LOSS_FUNCTION = 'categorical_crossentropy'
CLASSES = ['cn', 'emci', 'lmci']
NUM_CLASSES = len(CLASSES)

# --- 2. Data Loading ---
filepaths, labels = [], []
for cls in CLASSES:
    class_dir = os.path.join(base_dir, cls)
    if not os.path.exists(class_dir): continue
    for filename in os.listdir(class_dir):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(class_dir, filename))
            labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})

# --- 3. Functions for Training and Evaluation ---
def build_model():
    base_model = EfficientNetV2B0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions_layer = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions_layer)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=LOSS_FUNCTION, metrics=['accuracy'])
    return model

def plot_training_history(history, cycle_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].plot(history.history['accuracy'], color=COLORS['purple'], linewidth=2, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Accuracy')
    axes[0].set_title(f'Model Accuracy ({cycle_name})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['loss'], color=COLORS['purple'], linewidth=2, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'], linestyle='--', linewidth=2, label='Validation Loss')
    axes[1].set_title(f'Model Loss ({cycle_name})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def plot_multi_class_roc(y_true, y_pred_proba, cycle_name):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"], tpr["macro"], roc_auc["macro"] = all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})', color='navy', linestyle=':', linewidth=4)
    colors = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'ROC of class {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12); plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'Multi-Class ROC Curve ({cycle_name})', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.show()

def run_training_and_evaluation(train_df, val_df, test_df, cycle_name):
    print(f"\n{'='*40}\n  STARTING CYCLE: {cycle_name}\n{'='*40}")

    # Generator that ONLY applies the necessary preprocessing. No augmentation.
    datagen = ImageDataGenerator(preprocessing_function=efficientnet_preprocess)

    train_generator = datagen.flow_from_dataframe(
        dataframe=train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=True
    )
    validation_generator = datagen.flow_from_dataframe(
        dataframe=val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )
    test_generator = datagen.flow_from_dataframe(
        dataframe=test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASSES, shuffle=False
    )

    model = build_model()
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    history = model.fit(train_generator, epochs=EPOCHS, validation_data=validation_generator, callbacks=[early_stopping], verbose=1)

    print(f"\n--- Performance on Test Set ({cycle_name}) ---")
    predictions_prob = model.predict(test_generator)
    y_pred = np.argmax(predictions_prob, axis=1)
    y_true = test_generator.classes

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'Confusion Matrix ({cycle_name})', fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.show()

    plot_training_history(history, cycle_name)
    plot_multi_class_roc(y_true, predictions_prob, cycle_name)
    return model

# --- 4. Main Execution Logic ---
if __name__ == "__main__":
    if df.empty:
        raise ValueError("Dataset is empty. Check data paths.")

    print("\n--- Dataset Breakdown ---")
    print("1. Original Image Counts (Imbalanced):")
    original_counts = df['label'].value_counts()
    print(original_counts.to_string())
    print("-" * 35)

    # Step 1: Balancing by Downsampling to the smallest class
    min_samples = original_counts.min()
    balanced_df = pd.concat([
        df[df['label'] == cls].sample(n=min_samples, random_state=42) for cls in CLASSES
    ]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"2. After Balancing (Downsampling all classes to {min_samples} samples):")
    print(f"  - Total balanced samples: {len(balanced_df)} images")
    print(balanced_df['label'].value_counts().to_string())
    print("-" * 35)

    # Step 2: Splitting the balanced data into train, validation, and test sets
    train_val_df, test_df = train_test_split(balanced_df, test_size=0.20, random_state=42, stratify=balanced_df['label'])
    train_df, val_df = train_test_split(train_val_df, test_size=0.20, random_state=42, stratify=train_val_df['label'])

    print("3. Final Dataset Sizes for Model:")
    print(f"  - Training Set:   {len(train_df)} images")
    print(train_df['label'].value_counts().to_string())
    print(f"\n  - Validation Set: {len(val_df)} images")
    print(val_df['label'].value_counts().to_string())
    print(f"\n  - Test Set:         {len(test_df)} images")
    print(test_df['label'].value_counts().to_string())
    print("-" * 35)

    # --- Run Training Cycle ---
    original_model = run_training_and_evaluation(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        cycle_name="Training on Original Balanced Data"
    )

    # --- Model Saving ---
    save_dir = "/content/drive/MyDrive/summer_2025_research/bestmodel/EfficientNetV2B0_tri"
    os.makedirs(save_dir, exist_ok=True)
    model_save_path = os.path.join(save_dir, "TRAINING_WITH_ORIGINAL_BALANCED_DATA.keras")
    original_model.save(model_save_path)
    print(f"\n Model saved successfully to: {model_save_path}")